# SHAP Analysis: Understanding XGBoost Fraud Detection

This notebook provides interactive SHAP (SHapley Additive exPlanations) analysis for our fraud detection models.

**Goals:**
1. Understand which features drive fraud predictions
2. Compare feature importance across model variants (baseline vs hybrid)
3. Investigate false positives and false negatives
4. Identify why GNN embeddings don't improve performance

**Prerequisites:**
- Run training with `--save-models` flag to generate model artifacts
- Example: `python -m src.cli train-baseline --save-models`


In [2]:
import sys
import os
import glob
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import SHAP
import shap

# Add project root to path
sys.path.append('..')

from src.utils.explainability import (
    ModelExplainer, 
    load_saved_model, 
    compare_models,
    save_explainer
)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
shap.initjs()

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/dylan/Repos/ppa-fraud-notebook/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <B111F8D5-6AC6-3245-A6B5-94693F6992AB> /Users/dylan/Repos/ppa-fraud-notebook/.venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew

## 1. Load Models

Load saved models from training runs. You can choose which model type and window to analyze.


In [ ]:
# Configuration
MODEL_TYPE = "baseline"  # Options: baseline, graph_baseline, hybrid_hgt, hybrid_sage, etc.
WINDOW_IDX = -1  # -1 for latest, or specify window index

# Find available models
models_dir = f"../artifacts/models/{MODEL_TYPE}"
model_files = sorted(glob.glob(os.path.join(models_dir, "model_window_*.pkl")))

print(f"Found {len(model_files)} model windows in {models_dir}")
if model_files:
    for i, f in enumerate(model_files[-5:]):  # Show last 5
        print(f"  {os.path.basename(f)}")
else:
    print("\n⚠️ No models found! Run training with --save-models flag first.")
    print(f"Example: python -m src.cli train-baseline --save-models")


In [ ]:
# Load the selected model
if not model_files:
    raise FileNotFoundError("No model files available. Please train models first.")

model_path = model_files[WINDOW_IDX]
print(f"Loading: {os.path.basename(model_path)}")

model_bundle = load_saved_model(model_path)

# Display model info
print("\nModel Information:")
print(f"  Window: {model_bundle['window_info']['window_start']} to {model_bundle['window_info']['window_end']}")
print(f"  Train size: {model_bundle['window_info']['train_size']}")
print(f"  Test size: {model_bundle['window_info']['test_size']}")
print(f"  Features: {len(model_bundle['features'])}")
print(f"\nPerformance:")
for k, v in model_bundle['metrics'].items():
    if k != 'window_start':
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


## 2. Create SHAP Explainer

Initialize the SHAP explainer and compute SHAP values for all test instances.


In [ ]:
# Create explainer
explainer = ModelExplainer(
    model=model_bundle['model'],
    feature_names=model_bundle['features'],
    X_test=model_bundle['X_test'],
    y_test=model_bundle['y_test'],
    y_pred=model_bundle['y_pred'],
    model_type=MODEL_TYPE,
    window_info=model_bundle['window_info']
)

print(f"✓ SHAP explainer created")
print(f"  SHAP values shape: {explainer.shap_values.shape}")
print(f"  Expected value (base): {explainer.expected_value:.4f}")


## 3. Global Feature Importance

### 3.1 SHAP Summary Plot
Shows how each feature contributes to predictions across all instances.


In [ ]:
# Summary plot (beeswarm)
fig = explainer.plot_summary(max_display=20, aggregate_embeddings=True)
plt.show()


### 3.2 Feature Importance Bar Chart
Mean absolute SHAP values showing overall feature importance.


In [ ]:
fig = explainer.plot_feature_importance_bar(max_display=20, aggregate_embeddings=True)
plt.show()


### 3.3 Feature Importance Table


In [ ]:
importance_df = explainer.get_feature_importance_df(aggregate_embeddings=True)
print("Top 15 Most Important Features:")
importance_df.head(15)


## 4. Feature Dependence Analysis

Understand how individual features affect predictions and their interactions.


In [ ]:
# Select top features for dependence analysis
top_features = importance_df[importance_df['feature'] != 'GNN_embeddings'].head(3)['feature'].tolist()
print(f"Analyzing dependence for: {top_features}")

for feature in top_features:
    fig = explainer.plot_dependence(feature)
    plt.show()


## 5. Local Explanations: Individual Predictions

### 5.1 Understand False Positives
Why did the model incorrectly flag legitimate listings as fraud?


In [ ]:
# Count prediction types
pred_counts = explainer.df_test['prediction_type'].value_counts()
print("Prediction Distribution:")
print(pred_counts)
print(f"\nFalse Positive Rate: {pred_counts.get('FP', 0) / (pred_counts.get('TN', 0) + pred_counts.get('FP', 0)):.2%}")
print(f"False Negative Rate: {pred_counts.get('FN', 0) / (pred_counts.get('TP', 0) + pred_counts.get('FN', 0)):.2%}")


In [ ]:
# Explain top false positives
fp_data = explainer.df_test[explainer.df_test['prediction_type'] == 'FP'].sort_values('y_pred', ascending=False)

if len(fp_data) > 0:
    print(f"Analyzing top 5 false positives out of {len(fp_data)} total")
    for i in range(min(5, len(fp_data))):
        idx = fp_data.index[i]
        print(f"\n--- False Positive {i+1} (Index: {idx}, Score: {fp_data.iloc[i]['y_pred']:.4f}) ---")
        fig = explainer.plot_waterfall(idx, aggregate_embeddings=True)
        plt.show()
else:
    print("No false positives in this window!")


### 5.2 Compare True Positives vs False Positives
What distinguishes correct fraud detections from false alarms?


In [ ]:
fig = explainer.analyze_cohort_differences('TP', 'FP', top_n_features=15)
if fig:
    plt.show()
else:
    print("Insufficient data for cohort comparison")


## 6. Model Comparison: Baseline vs Hybrid

**Key Question:** Why doesn't the hybrid model (with GNN embeddings) outperform the baseline?

Let's compare feature importance to see if embeddings are actually useful.


In [ ]:
# Load models for comparison
MODELS_TO_COMPARE = ["baseline", "hybrid_hgt"]  # Adjust based on what you've trained
COMPARISON_WINDOW_IDX = -1  # Use same window for fair comparison

explainers_dict = {}

for model_type in MODELS_TO_COMPARE:
    models_dir = f"../artifacts/models/{model_type}"
    model_files = sorted(glob.glob(os.path.join(models_dir, "model_window_*.pkl")))
    
    if not model_files:
        print(f"⚠️ No models found for {model_type}. Skipping.")
        continue
    
    model_path = model_files[COMPARISON_WINDOW_IDX]
    print(f"Loading {model_type}: {os.path.basename(model_path)}")
    
    bundle = load_saved_model(model_path)
    
    exp = ModelExplainer(
        model=bundle['model'],
        feature_names=bundle['features'],
        X_test=bundle['X_test'],
        y_test=bundle['y_test'],
        y_pred=bundle['y_pred'],
        model_type=model_type,
        window_info=bundle['window_info']
    )
    
    explainers_dict[model_type] = exp
    
    print(f"  AUC-PR: {bundle['metrics']['auc_pr']:.4f}")
    print(f"  P@100: {bundle['metrics']['p@100']:.4f}")

print(f"\n✓ Loaded {len(explainers_dict)} models for comparison")


In [ ]:
# Generate comparison plot
if len(explainers_dict) >= 2:
    fig = compare_models(explainers_dict, top_n_features=15, aggregate_embeddings=True)
    plt.show()
else:
    print("Need at least 2 models to compare. Train additional models with --save-models flag.")


### 6.1 Embedding Contribution Analysis

For hybrid models, let's specifically examine how much the GNN embeddings contribute.


In [ ]:
# Check if we have a hybrid model loaded
hybrid_explainer = None
for name, exp in explainers_dict.items():
    if 'hybrid' in name:
        hybrid_explainer = exp
        hybrid_name = name
        break

if hybrid_explainer:
    print(f"Analyzing embedding contribution in {hybrid_name}")
    
    # Get embedding features
    embed_features = hybrid_explainer._get_embedding_features()
    print(f"\nFound {len(embed_features)} embedding features")
    
    # Calculate total embedding contribution
    embed_indices = [i for i, name in enumerate(hybrid_explainer.feature_names) if name in embed_features]
    embed_shap = hybrid_explainer.shap_values[:, embed_indices]
    
    # Total embedding SHAP per instance
    total_embed_shap = np.sum(embed_shap, axis=1)
    
    # Statistics
    print(f"\nEmbedding Contribution Statistics:")
    print(f"  Mean absolute embedding SHAP: {np.abs(embed_shap).mean():.6f}")
    print(f"  Mean absolute total SHAP: {np.abs(hybrid_explainer.shap_values).mean():.6f}")
    print(f"  Embedding contribution: {np.abs(embed_shap).mean() / np.abs(hybrid_explainer.shap_values).mean() * 100:.2f}%")
    
    # Distribution plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(np.abs(total_embed_shap), bins=50, alpha=0.7, label='Absolute Embedding SHAP')
    ax.set_xlabel('|Total Embedding SHAP|')
    ax.set_ylabel('Frequency')
    ax.set_title('Distribution of Embedding Contribution Across Test Set')
    ax.legend()
    plt.show()
else:
    print("No hybrid model found. Train a hybrid model with: python -m src.cli train-hybrid --save-models")


## 7. Export Results

Save all visualizations and feature importance tables for reporting.


In [ ]:
# Export all SHAP visualizations for the current model
if 'explainer' in locals():
    window_idx = model_bundle['window_info']['window_idx']
    output_dir = f"../artifacts/shap/{MODEL_TYPE}/window_{window_idx}"
    
    print(f"Saving all SHAP visualizations to {output_dir}...")
    save_explainer(explainer, output_dir, prefix="")
    print("✓ Export complete!")
else:
    print("No explainer loaded. Run the cells above first.")


## Next Steps

1. **Compare multiple windows**: Run this analysis across different time windows to see if feature importance is stable
2. **Deep dive on embeddings**: If embeddings are weak, investigate:
   - Graph structure quality (sparsity, connectivity)
   - GNN architecture (try different models: GAT, SAGE, HGT)
   - Embedding dimensionality (try 64, 128, 256)
3. **Feature engineering**: Based on top features, create interactions or derived features
4. **Threshold tuning**: Use SHAP insights to set better classification thresholds
